# PycWB Tutorial

![core workflow](./img/core-workflow.png)

## 1. Run your first example

First, we download the example user parameter file

In [ ]:
import os

import pycwb
from pycwb.config import Config
from pycwb.modules.logger import logger_init
import matplotlib.pyplot as plt

logger_init()

config = Config()
config.load_from_yaml('./user_parameters.yaml')

generate injected data for each detector with given parameters in config

In [ ]:
# from pycwb.modules.read_data import read_from_catalog, read_from_online
# from gwpy.timeseries import TimeSeries

# import requests
# from gwosc.locate import get_urls
t0 = 1126259462.4

# data = []
# for ifo in config.ifo:
#   url = get_urls(ifo, t0, t0)[-1]

#   print('Downloading: ' , url)
#   fn = os.path.basename(url)
#   with open(fn,'wb') as strainfile:
#       straindata = requests.get(url)
#       strainfile.write(straindata.content)

#   strain = TimeSeries.read(fn,format='hdf5.gwosc')
#   d = strain.crop(t0-150, t0+150)
#   d_resampled = d.resample(2048)
#   data.append(d_resampled)

import pickle
with open('../../data/frame_data.pkl', 'rb') as f:
    data = pickle.load(f)

## apply data conditioning to the data

In cWB, data conditioning includes line removal and whitening. The whitened data and the noise level for each pixel (nRMS) are returned.

In [ ]:
from pycwb.modules.data_conditioning import data_conditioning
from pycwb.modules.plot import plot_spectrogram

strains, nRMS = data_conditioning(config, data)
strains, nRMS

Since we already know where the signal is, let's do a little cheating first ...

In [ ]:
%matplotlib inline
from wdm_wavelet.wdm import WDM

wdm = WDM(32, 64, 6, 10)

sliced_data = strains[0].data.crop(145, 145)

tf_map = wdm.t2w(sliced_data)

fig, ax = plt.subplots(2,1, figsize=(6, 6))
ax[0].plot(sliced_data.sample_times, sliced_data)
ax[0].set_xlim(t0-1, t0+1)
ax[0].set_ylabel('Strain')
ax[0].grid(False)
tf_map.plot_energy(fig=fig, ax=ax[1], colorbar=False)
ax[1].set_xlim(t0-1, t0+1)
ax[1].set_ylim(10, 800)
plt.show()
plt.close()

## calculate coherence

![alt text](img/MRA.png)

In [ ]:
from pycwb.modules.coherence import coherence

# calculate coherence
fragment_clusters = coherence(config, strains, nRMS)

In [ ]:
from pprint import pprint
pprint(fragment_clusters[0][0])

In [ ]:
from pprint import pprint
pprint(fragment_clusters[0][0].clusters[0].pixels[0])

## supercluster

In [ ]:
from pycwb.modules.super_cluster import supercluster
from pycwb.types.network import Network

network = Network(config, strains, nRMS)

pwc_list = supercluster(config, network, fragment_clusters, strains)

In [ ]:
%matplotlib inline
from gwpy.spectrogram import Spectrogram

for cluster in pwc_list[0].clusters:
    merged_map, start, dt, df = cluster.get_sparse_map("likelihood")

    plt = Spectrogram(merged_map, t0=start, dt=dt, f0=0, df=df).plot()
    plt.colorbar()
plt.show()


In [ ]:
plt.close()

## Likelihood

In [ ]:
from pycwb.modules.likelihood import likelihood

events, clusters, skymap_statistics = likelihood(config, network, pwc_list)

plot statistics

In [ ]:
%matplotlib inline
from pycwb.modules.plot import plot_event_on_spectrogram

plt = plot_event_on_spectrogram(strains[0], events)
plt.show()

In [ ]:
%matplotlib inline
from gwpy.spectrogram import Spectrogram

plt.close()
for cluster in clusters:
    merged_map, start, dt, df = cluster.get_sparse_map("likelihood")

    plt = Spectrogram(merged_map, t0=start, dt=dt, f0=0, df=df).plot()
    plt.colorbar()

In [ ]:
%matplotlib inline
from gwpy.spectrogram import Spectrogram

plt.close()
for cluster in clusters:
    merged_map, start, dt, df = cluster.get_sparse_map("null")

    plt = Spectrogram(merged_map, t0=start, dt=dt, f0=0, df=df).plot()
    plt.colorbar()
plt.show()

In [ ]:
from pycwb.modules.reconstruction import get_network_MRA_wave
from pycwb.modules.plot.waveform import plot_reconstructed_waveforms
from matplotlib import pyplot as plt

event = events[0]
cluster = clusters[0]
reconstructed_waves = get_network_MRA_wave(config, cluster, config.rateANA, config.nIFO, config.TDRate,
                                               'signal', 0, True)
for reconstructed_wave in reconstructed_waves:
  plt.plot(reconstructed_wave.sample_times, reconstructed_wave.data)
plt.xlim((event.left[0], event.left[0] + event.stop[0] - event.start[0]))

In [ ]:
pprint(events[0])